# 2 - Retrieval Evaluation

We compare **three retrieval approaches** against the ground truth and use
the metrics from module 4:

| Approach | How |
|---|---|
| Full-text search | Postgres `tsvector` / `ts_rank` over the chunk text |
| Vector search | pgvector cosine distance, `text-embedding-3-small` (what the live n8n flow uses) |
| Hybrid + RRF | Reciprocal Rank Fusion of the two rankings (module 6: hybrid search + document re-ranking) |

**Metrics** (top 5 results, evaluated on two levels):

- **Hit Rate** - fraction of questions where the correct chunk/page is in the top 5
- **MRR** (Mean Reciprocal Rank) - how high up the correct result appears
- *chunk level* = the exact chunk the question came from; *doc level* = the right documentation page

Requires the Docker stack running (`make up` + `make ingest`), because we query
the same pgvector table the production flow uses.

In [1]:
import json
import os

import numpy as np
import pandas as pd
import psycopg
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

load_dotenv("../.env")

client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"

conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    autocommit=True,
)
conn.execute("SELECT count(*) FROM n8n_vectors").fetchone()

(589,)

In [2]:
df_gt = pd.read_csv("../data/ground-truth.csv")
ground_truth = df_gt.to_dict(orient="records")
len(ground_truth)

450

## Embed all questions once

All three approaches are evaluated on the same questions, so we embed them
once in batches instead of once per approach.

In [3]:
def embed_batch(texts, batch_size=100):
    vectors = []
    for i in tqdm(range(0, len(texts), batch_size)):
        response = client.embeddings.create(
            model=EMBED_MODEL, input=texts[i:i + batch_size]
        )
        vectors.extend(d.embedding for d in response.data)
    return vectors


question_vectors = embed_batch(df_gt.question.tolist())
len(question_vectors)

  0%|          | 0/5 [00:00<?, ?it/s]

450

## The three search functions

All of them return `[(chunk_id, doc_id), ...]` so they are interchangeable.

In [4]:
def text_search(query, k=5):
    # plainto_tsquery ANDs all words - full sentences would match almost
    # nothing. We switch to OR semantics (any word may match, ts_rank
    # still rewards matching more words) for a fair comparison.
    rows = conn.execute(
        '''
        SELECT metadata->>'chunk_id', metadata->>'doc_id'
        FROM n8n_vectors,
             to_tsquery('english',
                 replace(plainto_tsquery('english', %s)::text, ' & ', ' | ')
             ) AS query
        WHERE to_tsvector('english', text) @@ query
        ORDER BY ts_rank(to_tsvector('english', text), query) DESC
        LIMIT %s
        ''',
        (query, k),
    ).fetchall()
    return rows


def vector_search(query_vector, k=5):
    rows = conn.execute(
        '''
        SELECT metadata->>'chunk_id', metadata->>'doc_id'
        FROM n8n_vectors
        ORDER BY embedding <=> %s::vector
        LIMIT %s
        ''',
        (str(query_vector), k),
    ).fetchall()
    return rows


def hybrid_search_rrf(query, query_vector, k=5, k_rrf=60, fetch=10):
    '''Reciprocal Rank Fusion: score = sum over rankings of 1/(k_rrf + rank).'''
    rankings = [text_search(query, k=fetch), vector_search(query_vector, k=fetch)]
    scores = {}
    for ranking in rankings:
        for rank, row in enumerate(ranking):
            scores[row] = scores.get(row, 0) + 1 / (k_rrf + rank + 1)
    fused = sorted(scores, key=scores.get, reverse=True)
    return fused[:k]

In [5]:
q = ground_truth[0]
print("Q:", q["question"])
print("expected:", q["chunk_id"])
vector_search(question_vectors[0])

Q: How do AI agents differ from AI chains in n8n when it comes to keeping track of earlier conversation context?
expected: build/integrate-ai/understand-ai-components/agents-vs-chains#1


[('build/integrate-ai/understand-ai-components/what-chains-do#2',
  'build/integrate-ai/understand-ai-components/what-chains-do'),
 ('build/integrate-ai/understand-ai-components/agents-vs-chains#1',
  'build/integrate-ai/understand-ai-components/agents-vs-chains'),
 ('build/integrate-ai/understand-ai-components/what-chains-do#1',
  'build/integrate-ai/understand-ai-components/what-chains-do'),
 ('build/integrate-ai/understand-ai-components/how-memory-works#1',
  'build/integrate-ai/understand-ai-components/how-memory-works'),
 ('build/integrate-ai/ai-examples#2', 'build/integrate-ai/ai-examples')]

## Metrics (module 4)

In [6]:
def hit_rate(relevance_total):
    return sum(any(line) for line in relevance_total) / len(relevance_total)


def mrr(relevance_total):
    total = 0.0
    for line in relevance_total:
        for rank, is_relevant in enumerate(line):
            if is_relevant:
                total += 1 / (rank + 1)
                break
    return total / len(relevance_total)


def evaluate(search_results_per_question):
    '''search results: list of [(chunk_id, doc_id), ...] per GT record.'''
    rel_chunk, rel_doc = [], []
    for gt, results in zip(ground_truth, search_results_per_question):
        rel_chunk.append([cid == gt["chunk_id"] for cid, _ in results])
        rel_doc.append([did == gt["doc_id"] for _, did in results])
    return {
        "hit_rate_chunk": hit_rate(rel_chunk),
        "mrr_chunk": mrr(rel_chunk),
        "hit_rate_doc": hit_rate(rel_doc),
        "mrr_doc": mrr(rel_doc),
    }

## Run all three evaluations

In [7]:
results_text = [
    text_search(gt["question"]) for gt in tqdm(ground_truth)
]
results_vector = [
    vector_search(vec) for vec in tqdm(question_vectors)
]
results_hybrid = [
    hybrid_search_rrf(gt["question"], vec)
    for gt, vec in tqdm(zip(ground_truth, question_vectors), total=len(ground_truth))
]

  0%|          | 0/450 [00:00<?, ?it/s]

  0%|          | 0/450 [00:00<?, ?it/s]

  0%|          | 0/450 [00:00<?, ?it/s]

In [8]:
df_results = pd.DataFrame([
    {"approach": "full-text (ts_rank)", **evaluate(results_text)},
    {"approach": "vector (pgvector)", **evaluate(results_vector)},
    {"approach": "hybrid + RRF", **evaluate(results_hybrid)},
]).set_index("approach")
df_results.round(3)

,hit_rate_chunk,mrr_chunk,hit_rate_doc,mrr_doc
approach,,,,
full-text (ts_rank),0.653,0.472,0.776,0.599
vector (pgvector),0.904,0.781,0.936,0.861
hybrid + RRF,0.889,0.672,0.924,0.763


## Conclusion

**Vector search wins on every metric** (hit rate 0.90 / MRR 0.78 at chunk
level, 0.94 / 0.86 at page level) and is what the live n8n flow uses via the
PGVector *retrieve-as-tool* node - offline evaluation and production agree.

- Full-text search alone lags far behind (0.65 / 0.47): questions are
  paraphrased, so keyword overlap with the docs is weak.
- Hybrid + RRF (0.89 / 0.67) does *not* beat pure vector here - fusing in the
  weaker text ranking mostly dilutes the vector ranking. It would earn its
  keep with query types where exact tokens matter (node names, error
  strings); worth revisiting once real user traffic accumulates.
